# LangGraph 核心用法（带看懂版）

> 目标：带你看懂 LangGraph 的核心代码。全书用同一条真实场景贯穿：**电商订单处理**。
> 先看一张总览图，再用订单系统串起全部核心概念，看懂两种写死方式，最后逐个看懂六大核心能力。
> 基于 langgraph 1.2.11（2026-08-18）验证。

## 框架总览：一张图看懂 LangGraph

LangGraph 的核心是把流程画成一张有向图：**状态在节点之间流动，边决定流转方向，条件边决定分支**。
图上的节点可以**循环**、可以**并行**，执行进度可以被 **Checkpointer 持久化**，
流程可以 **interrupt 挂起等人审批**再恢复，还可以用 **stream 逐步输出**，多 Agent 则是把整个图当成一个节点。

```mermaid
flowchart TB
    A[用户输入] -->|invoke| B[State（状态）]
    B --> C[Node（节点）]
    C --> D{条件边}
    D -->|继续| E[下一个节点]
    D -->|结束| Z[END（结束）]
    E -.循环回到某节点.-> C
    C -.并行 Send.-> F[并行节点组]
    C -.checkpoint 落盘.-> G[Checkpointer（检查点）]
    G -.崩溃后按 thread 恢复.-> C
    C -.interrupt 挂起.-> H[人工审批]
    H -.resume 恢复.-> C
```

## 核心概念（先读一遍，遇到再回来看）

| 概念 | 作用 | 在哪看 |
| --- | --- | --- |
| State | 图的共享状态，用 `TypedDict` 定义 | 概念关系、例子一 |
| Node | 普通函数，返回 state 的部分更新 | 概念关系、例子一 |
| Edge / 条件边 | 节点流转与分支路由 | 概念关系、例子一 |
| 循环 | 条件边指回自己，形成 Loop | 能力 1 |
| Send | 一个节点扇出多个并行子节点 | 能力 2 |
| Checkpointer | 执行进度持久化，按 thread 恢复 | 能力 3 |
| interrupt | 流程挂起，等外部（人）输入再恢复 | 能力 4 |
| stream | 逐步拿到每一步的输出 | 能力 5 |
| 多 Agent 编排 | supervisor 分派任务给多个 worker | 能力 6 |

## 概念之间的关系：用订单系统全部串起来

下面把上一张表里的**全部概念**放进同一个订单处理系统。先看全景图：

```mermaid
flowchart TB
    A[receive_order（接收订单）] --> B{route_by_stock（库存充足？）}
    B -->|有货| C[deduct（扣库存）]
    B -->|缺货| D[restock（通知补货）]
    C --> E[complete（完成订单）]
    D --> E
    E --> F[END（结束）]
    A -.并行 Send.-> P[订单/库存/物流 同时查询]
    P -.查询结果.-> A
    C -.物流失败循环重试.-> L[query（查询物流）]
    L -.查询成功.-> C
    C -.checkpoint 落盘.-> G[Checkpointer（检查点）]
    C -.interrupt 挂起.-> H[人工审批（大额订单）]
    H -.resume 恢复.-> C
    E -.stream 逐步输出.-> S[前端展示]
    S -.多 Agent 分派.-> M[supervisor（主 Agent）]
```

| 概念 | 在订单系统里的位置 |
| --- | --- |
| State | `OrderState`：order_id、item、in_stock、logs，订单数据全程跟着图走 |
| Node | `receive_order`、`deduct`、`restock`、`complete` 等处理环节 |
| Edge / 条件边 | `route_by_stock` 决定有货扣库存、缺货通知补货 |
| 循环 | 物流查询失败自动重试，直到成功（能力 1） |
| Send | 订单、库存、物流三个查询并行执行（能力 2） |
| Checkpointer | 批量订单处理进度落盘，崩溃后按 thread 恢复（能力 3） |
| interrupt | 大额订单挂起等人审批，resume 后继续（能力 4） |
| stream | 前端逐步展示每一步处理过程（能力 5） |
| 多 Agent | supervisor 把任务分派给订单 Agent / 客服 Agent（能力 6） |

先运行下面的**骨架流水线**（只有 State、Node、Edge、条件边、Reducer），建立最小直觉；其余能力会在后面的「核心能力逐个看」用同一个订单场景逐个跑。

In [ ]:
from typing import Annotated, TypedDict
import operator

from langgraph.graph import StateGraph, START, END


class OrderState(TypedDict):
    order_id: str
    item: str
    in_stock: bool
    logs: Annotated[list[str], operator.add]


def receive_order(state: OrderState) -> dict:
    """接收订单。"""
    return {"logs": [f"收到订单：{state['order_id']}（{state['item']}）"]}


def deduct(state: OrderState) -> dict:
    """有货：扣减库存。"""
    return {"logs": ["库存已扣减"]}


def restock(state: OrderState) -> dict:
    """缺货：通知补货。"""
    return {"logs": ["库存不足，已通知补货"]}


def complete(state: OrderState) -> dict:
    """完成订单。"""
    return {"logs": ["订单完成"]}


def route_by_stock(state: OrderState) -> str:
    """按库存是否充足路由。"""
    return "deduct" if state["in_stock"] else "restock"


graph = StateGraph(OrderState)
graph.add_node("receive_order", receive_order)
graph.add_node("deduct", deduct)
graph.add_node("restock", restock)
graph.add_node("complete", complete)
graph.add_edge(START, "receive_order")
graph.add_conditional_edges(
    "receive_order", route_by_stock, {"deduct": "deduct", "restock": "restock"}
)
graph.add_edge("deduct", "complete")
graph.add_edge("restock", "complete")
graph.add_edge("complete", END)
compiled = graph.compile()

for order in (
    {"order_id": "A001", "item": "键盘", "in_stock": True, "logs": []},
    {"order_id": "A002", "item": "显示器", "in_stock": False, "logs": []},
):
    result = compiled.invoke(order)
    print(result["logs"])


### 骨架流水线：按执行顺序走一遍

用订单 `{"order_id": "A001", "item": "键盘", "in_stock": True, "logs": []}` 追踪：

| 顺序 | 执行的代码 | 此时 logs 变成 |
| --- | --- | --- |
| 1 | `compiled.invoke(订单)` | `[]` |
| 2 | 进入 `receive_order` | `["收到订单：A001（键盘）"]` |
| 3 | `route_by_stock`：`in_stock=True` → `"deduct"` | 不变 |
| 4 | 进入 `deduct` | `["收到订单：A001（键盘）", "库存已扣减"]` |
| 5 | 进入 `complete` | `["收到订单：A001（键盘）", "库存已扣减", "订单完成"]` |
| 6 | 到达 END，返回 | 打印三条记录 |

订单 `A002`（`in_stock=False`）只有第 3、4 步不同：走 `restock`，输出
`["收到订单：A002（显示器）", "库存不足，已通知补货", "订单完成"]`。

这张骨架图只演示了 State、Node、Edge、条件边、Reducer 五个概念；循环、并行、持久化、人工介入、流式、多 Agent 是加在骨架上的增强，后面「核心能力逐个看」会逐一跑给你看。

## 先看懂两种写死方式

LangGraph 不会自动生成业务路径，流程必须由人定义。例子一和例子二就是两种写死方式：
例子一用图 API 写死（节点、边、条件边），例子二用函数体写死（语句顺序、if 分支）。
两个例子做同一个任务：**计算订单运费**——金额 ≥ 99 元免运费，否则收 10 元。

### 例子一：手搓 StateGraph（最底层、控制力最强）

先看这张图，再看下面的代码：

```mermaid
flowchart LR
    A[START] --> B[classify 记录入口]
    B --> C{route_by_amount<br/>金额 >= 99 ？}
    C -->|是| D[handle_free 免运费]
    C -->|否| E[handle_paid 收 10 元]
    D --> F[END]
    E --> F
```

代码里的每个节点、边、条件边都能在这张图上找到。

In [ ]:
from typing import Annotated, TypedDict
import operator

from langgraph.graph import StateGraph, START, END


class FeeState(TypedDict):
    order_amount: float
    shipping_fee: float
    steps: Annotated[list[str], operator.add]


def classify(state: FeeState) -> dict:
    """记录入口节点。"""
    return {"steps": ["classify"]}


def handle_free(state: FeeState) -> dict:
    """免运费分支。"""
    return {"shipping_fee": 0.0, "steps": ["免运费"]}


def handle_paid(state: FeeState) -> dict:
    """收运费分支。"""
    return {"shipping_fee": 10.0, "steps": ["运费 10 元"]}


def route_by_amount(state: FeeState) -> str:
    """金额 >= 99 免运费，否则收运费。"""
    return "free" if state["order_amount"] >= 99 else "paid"


def build_graph():
    """构建并编译计算运费的状态图。"""
    graph = StateGraph(FeeState)
    graph.add_node("classify", classify)
    graph.add_node("free", handle_free)
    graph.add_node("paid", handle_paid)
    graph.add_edge(START, "classify")
    graph.add_conditional_edges(
        "classify",
        route_by_amount,
        {"free": "free", "paid": "paid"},
    )
    graph.add_edge("free", END)
    graph.add_edge("paid", END)
    return graph.compile()


compiled = build_graph()
for amount in (129.0, 59.0):
    result = compiled.invoke({"order_amount": amount, "shipping_fee": 0.0, "steps": []})
    print(f"amount={amount} -> fee={result['shipping_fee']}, steps={result['steps']}")


#### 例子一：按执行顺序拆解

用 `order_amount=129.0` 走一遍上面的图：

| 顺序 | 执行的代码 | 此时 state 变成 |
| --- | --- | --- |
| 1 | `compiled.invoke({"order_amount": 129.0, "shipping_fee": 0.0, "steps": []})` | `shipping_fee: 0.0, steps: []` |
| 2 | 进入 `classify` | `steps: ["classify"]` |
| 3 | `route_by_amount`：`129 >= 99` → `"free"` | 不变 |
| 4 | 进入 `handle_free`：`shipping_fee = 0.0` | `shipping_fee: 0.0, steps: ["classify", "免运费"]` |
| 5 | 到达 END，返回 | 打印 `amount=129.0 -> fee=0.0` |

换成 `order_amount=59.0`：第 3 步返回 `"paid"`，第 4 步 `shipping_fee = 10.0`。
关键认知：**代码里先定义谁不重要，执行顺序由边决定**。

### 例子二：函数式 API（代码量少一半）

同一个任务。`@entrypoint()` 把函数包装成图（`main` 类型就是 `Pregel`），`@task` 标记子任务；函数体的行顺序就是执行顺序：

```mermaid
flowchart LR
    A[main.invoke 订单金额] --> B[组装 steps]
    B --> C[calc_fee 任务启动]
    C --> D[.result 等待结果]
    D --> E[return 输出运费]
```

In [ ]:
from langgraph.func import entrypoint, task


@task
def calc_fee(order_amount: float) -> float:
    """金额 >= 99 免运费，否则 10 元。"""
    return 0.0 if order_amount >= 99 else 10.0


@entrypoint()
def main(state: dict) -> dict:
    """函数式 API 版：计算订单运费。"""
    steps = ["classify", "免运费" if state["order_amount"] >= 99 else "运费 10 元"]
    fee = calc_fee(state["order_amount"]).result()
    return {"shipping_fee": fee, "steps": steps}


for amount in (129.0, 59.0):
    result = main.invoke({"order_amount": amount})
    print(f"amount={amount} -> fee={result['shipping_fee']}, steps={result['steps']}")


#### 例子二：按执行顺序拆解

用 `order_amount=129.0`：

| 顺序 | 执行的代码 | 结果 |
| --- | --- | --- |
| 1 | `main.invoke({"order_amount": 129.0})` | 开始执行入口函数 |
| 2 | 组装 `steps`：`129 >= 99` → `"免运费"` | `steps: ["classify", "免运费"]` |
| 3 | `calc_fee(129.0)`：启动 `@task` 任务 | 返回 Future |
| 4 | `.result()`：等待完成，得到 `0.0` | `fee = 0.0` |
| 5 | `return` | `invoke` 返回 |

区别一句话：例子一用**边**决定分支，例子二用**函数里的 if** 决定；逻辑顺序都由人写死，框架只负责调度与包装。

### 例子三：create_react_agent（预置 ReAct 循环）

需要 `OPENAI_API_KEY`，没 Key 先读懂调用链。它内置了「模型 → 工具 → 条件边 → 模型」的循环模板，**骨架写死，具体调哪个工具由模型动态决定**。这里让 Agent 帮你查库存。

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


@tool
def check_stock(item: str) -> str:
    """查询商品是否有货。"""
    return f"{item} 有货"


model = ChatOpenAI(model="gpt-4o-mini")
agent = create_react_agent(model, tools=[check_stock])
response = agent.invoke({"messages": [{"role": "user", "content": "键盘有货吗？"}]})
print(response["messages"][-1].content)


#### 例子三：按执行顺序拆解（ReAct 循环）

| 顺序 | 执行内容 | 说明 |
| --- | --- | --- |
| 1 | `agent.invoke(用户消息)` | 把「键盘有货吗？」发给模型 |
| 2 | 模型返回 tool_call：调用 `check_stock("键盘")` | 模型决定需要工具 |
| 3 | 框架执行工具，返回「键盘 有货」 | 结果喂回模型 |
| 4 | 模型返回最终回答 | 循环结束 |

它就是例子一的图加了一层循环：`模型 → 工具 → 条件边 → 模型`，直到模型不再要工具。

## 三种写法怎么选

| 写法 | 代码量 | 控制力 | 适用场景 |
| --- | --- | --- | --- |
| 手搓 StateGraph | 多 | 最强 | 复杂图、精确控制路由与状态 |
| entrypoint / task | 少 | 中 | 线性或简单分支流程 |
| create_react_agent | 最少 | 低 | 标准工具调用 Agent，快速交付 |

AI 会帮你写代码，但选哪种写法、什么时候需要精确控制路由，需要你能看懂代码才能判断。

## 核心能力逐个看

订单流水线和例子一、二都是「一条直线加一个分支」。下面这六个能力才是 LangGraph 区别于普通管道（LangChain Chain / 纯 Python）的地方，全部沿用订单场景。

### 能力 1：循环（物流查询重试）

条件边指回自己，节点就会反复执行，直到路由函数说停。
真实场景：物流接口不稳定，失败就重试，直到查询成功。

```mermaid
flowchart LR
    A[START] --> B[query 查询物流]
    B --> C{should_retry<br/>attempts < 3 ？}
    C -->|是| B
    C -->|否| D[END]
```

In [ ]:
from typing import Annotated, TypedDict
import operator

from langgraph.graph import StateGraph, START, END


class RetryState(TypedDict):
    attempts: int
    logs: Annotated[list[str], operator.add]


def query_logistics(state: RetryState) -> dict:
    """模拟物流查询：前 2 次失败，第 3 次成功。"""
    attempts = state["attempts"] + 1
    if attempts >= 3:
        return {"attempts": attempts, "logs": [f"第 {attempts} 次：查询成功"]}
    return {"attempts": attempts, "logs": [f"第 {attempts} 次：查询失败，准备重试"]}


def should_retry(state: RetryState) -> str:
    """还没成功就继续重试，否则结束。"""
    return "query" if state["attempts"] < 3 else "end"


graph = StateGraph(RetryState)
graph.add_node("query", query_logistics)
graph.add_edge(START, "query")
graph.add_conditional_edges("query", should_retry, {"query": "query", "end": END})
retry_graph = graph.compile()

result = retry_graph.invoke({"attempts": 0, "logs": []})
print(result["logs"])


循环的拆解：

- `query_logistics` 每次把 `attempts + 1`，前 2 次标记失败，第 3 次标记成功
- `should_retry` 返回 `"query"` 时，条件边把执行**指回 query 自己**；返回 `"end"` 时走 END
- 输出能看到完整重试轨迹：失败 → 失败 → 成功
- 这也是 ReAct 循环的底层：模型节点反复执行，直到不再调用工具

### 能力 2：并行（订单 / 库存 / 物流同时查询）

一个节点可以一次性派出多个子任务，LangGraph 并行执行它们，结果用 Reducer 收集。
真实场景：下单前同时查订单、库存、物流三个数据源。

```mermaid
flowchart LR
    A[START] --> B[dispatcher 派发]
    B --> C1[worker 订单]
    B --> C2[worker 库存]
    B --> C3[worker 物流]
    C1 --> D[results 汇总]
    C2 --> D
    C3 --> D
    D --> E[END]
```

In [ ]:
from typing import Annotated, TypedDict
import operator

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send, Command


class CheckState(TypedDict):
    results: Annotated[list[str], operator.add]


def dispatcher(state: CheckState):
    """同时派出订单、库存、物流三个查询。"""
    tasks = [
        Send("worker", {"source": "订单"}),
        Send("worker", {"source": "库存"}),
        Send("worker", {"source": "物流"}),
    ]
    return Command(goto=tasks)


def worker(state: dict) -> dict:
    """模拟一个数据源查询。"""
    return {"results": [f"{state['source']}查询完成"]}


graph = StateGraph(CheckState)
graph.add_node("dispatcher", dispatcher)
graph.add_node("worker", worker)
graph.add_edge(START, "dispatcher")
graph.add_edge("worker", END)
parallel_graph = graph.compile()

print(parallel_graph.invoke({"results": []})["results"])


并行的拆解：

- `dispatcher` 返回 `Command(goto=[Send(...), ...])`，每个 `Send` 指定目标节点和该任务的输入
- LangGraph 为每个 `Send` 启动一个并行的 `worker` 实例，互不阻塞
- 注意不要同时加 `dispatcher → worker` 静态边，`Send` 本身就是动态边
- `results` 用 `operator.add` 收集中间结果，最终一次性汇总

### 能力 3：Checkpointer 订单进度持久化

给图配一个 checkpointer 后，每一步执行进度都会落盘。同一个 `thread_id` 再次调用，state 从上一次结束的位置继续。真实场景：处理大量订单时崩溃，重启后从断点继续。

```mermaid
flowchart LR
    A[START] --> B[process_one 处理订单]
    B --> C[END]
    B -.每次执行结果写入.-> D[Checkpointer 落盘]
    D -.同 thread 再次 invoke 从断点继续.-> B
```

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END


class ProgressState(TypedDict):
    processed: int


def process_one(state: ProgressState) -> dict:
    """每次处理一个订单。"""
    return {"processed": state["processed"] + 1}


graph = StateGraph(ProgressState)
graph.add_node("process_one", process_one)
graph.add_edge(START, "process_one")
graph.add_edge("process_one", END)
checkpoint_graph = graph.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "order-batch-1"}}
print("第一次：", checkpoint_graph.invoke({"processed": 0}, config))
print("第二次：", checkpoint_graph.invoke({"processed": 0}, config))


持久化的拆解：

- 第一次 `invoke` 后，`processed=1` 被写进 checkpointer（按 `order-batch-1` 保存）
- 第二次用同一个 `thread_id`，输入虽然是 `0`，但框架从 checkpoint 恢复，接着累加成 `2`
- `MemorySaver` 是内存版；生产用 Sqlite/Postgres 版本，重启不丢

### 能力 4：interrupt 大额订单人工审批（human-in-the-loop）

流程在指定节点挂起，返回给外部（人）一个待办；外部决定后通过 `Command(resume=...)` 从断点继续。真实场景：订单金额超过 1000 元必须人工审批，审批人可能几天后才点。

```mermaid
flowchart LR
    A[START] --> B[request_approval 请求审批]
    B -->|interrupt 挂起| C[人工审批]
    C -->|resume 恢复| D[finalize 执行]
    D --> E[END]
```

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt


class ApprovalState(TypedDict):
    amount: int
    approved: str


def request_approval(state: ApprovalState) -> dict:
    """挂起流程，等人工审批。"""
    decision = interrupt({"question": f"订单金额 {state['amount']} 元，是否批准？"})
    return {"approved": decision}


def finalize(state: ApprovalState) -> dict:
    """审批通过后执行。"""
    return {"approved": state["approved"] + "，已执行"}


graph = StateGraph(ApprovalState)
graph.add_node("request_approval", request_approval)
graph.add_node("finalize", finalize)
graph.add_edge(START, "request_approval")
graph.add_edge("request_approval", "finalize")
graph.add_edge("finalize", END)
approval_graph = graph.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "order-approval-1"}}

result = approval_graph.invoke({"amount": 1000, "approved": ""}, config)
print("第一次调用（挂起）：", result)
print("中断事件：", result.get("__interrupt__"))

result = approval_graph.invoke(Command(resume="同意"), config)
print("恢复后结果：", result["approved"])


人工介入的拆解：

- `interrupt(...)` 让流程**挂起**：第一次 `invoke` 返回的 state 里带 `__interrupt__`，流程没有结束
- 外部（审批人）看完待办后，用 `Command(resume="同意")` 把决定传回
- 框架从挂起节点恢复，`decision` 拿到 `"同意"`，继续走 `finalize`
- 这就是 LangChain 管道做不到的：**流程可以停几天，然后接着跑**

### 能力 5：stream 流式展示处理过程

`invoke` 只给你最终结果；`stream` 让你逐步看到每一步的输出。真实场景：前端逐条展示物流查询重试的进度。

图还是能力 1 的循环图，区别只在调用方式：`invoke` 等全部跑完返回一个结果，`stream` 每跑一步就吐一个事件。

In [10]:
# 复用能力 1 的 retry_graph，按步输出每次查询的事件
for event in retry_graph.stream({"attempts": 0, "logs": []}):
    print(event)


{'query': {'attempts': 1, 'logs': ['第 1 次：查询失败，准备重试']}}
{'query': {'attempts': 2, 'logs': ['第 2 次：查询失败，准备重试']}}
{'query': {'attempts': 3, 'logs': ['第 3 次：查询成功']}}


流式的拆解：

- 默认 `stream_mode="updates"`，每个事件形如 `{"query": {"attempts": 1, "logs": [...]}}`
- 可以看到节点名字、每步的 state 更新，方便前端展示和调试
- 还有 `stream_mode="values"`、`"messages"` 等模式，按需选择

### 能力 6：多 Agent 编排（订单 Agent / 客服 Agent）

多 Agent 不是玄学：把几个 Agent 当作图的节点，用一个 supervisor 节点分派任务。真实场景里，worker 内部可以是 `create_react_agent` 这种带模型的 Agent。

```mermaid
flowchart LR
    A[START] --> B[supervisor 主 Agent]
    B --> C{route 按任务分派}
    C -->|订单| D[worker_a 订单 Agent]
    C -->|其他| E[worker_b 客服 Agent]
    D --> F[END]
    E --> F
```

In [11]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class TeamState(TypedDict):
    task: str
    result: str


def supervisor(state: TeamState) -> dict:
    """主 Agent：记录任务并开始分派。"""
    return {"result": f"supervisor 收到任务：{state['task']}"}


def worker_a(state: TeamState) -> dict:
    """订单 Agent。"""
    return {"result": f"订单 Agent 处理：{state['task']}"}


def worker_b(state: TeamState) -> dict:
    """客服 Agent。"""
    return {"result": f"客服 Agent 处理：{state['task']}"}


def route(state: TeamState) -> str:
    """按任务内容分派。"""
    return "worker_a" if state["task"].startswith("订单") else "worker_b"


graph = StateGraph(TeamState)
graph.add_node("supervisor", supervisor)
graph.add_node("worker_a", worker_a)
graph.add_node("worker_b", worker_b)
graph.add_edge(START, "supervisor")
graph.add_conditional_edges("supervisor", route, {"worker_a": "worker_a", "worker_b": "worker_b"})
graph.add_edge("worker_a", END)
graph.add_edge("worker_b", END)
team_graph = graph.compile()

print(team_graph.invoke({"task": "订单查询", "result": ""}))
print(team_graph.invoke({"task": "退款咨询", "result": ""}))


{'task': '订单查询', 'result': '订单 Agent 处理：订单查询'}
{'task': '退款咨询', 'result': '客服 Agent 处理：退款咨询'}


多 Agent 的拆解：

- `supervisor` 是主 Agent（真实场景里它用模型决定派活）
- 条件边按任务内容路由到 `worker_a`（订单 Agent）或 `worker_b`（客服 Agent）
- 每个 worker 内部可以是独立 Agent；把「编排结构」和「Agent 内部逻辑」分开，就是多 Agent 系统的基本形态

## 核心能力总览

| 能力 | 订单场景 | 关键 API |
| --- | --- | --- |
| 循环 | 物流查询失败重试 | `add_conditional_edges` |
| 并行 | 订单/库存/物流同时查询 | `Send` / `Command` |
| 持久化 | 批量订单进度落盘恢复 | `compile(checkpointer=...)` |
| 人工介入 | 大额订单审批挂起恢复 | `interrupt` / `Command(resume=...)` |
| 流式 | 前端逐步展示处理进度 | `stream()` |
| 多 Agent | supervisor 分派订单/客服 Agent | StateGraph + 条件边 |